# Baseline CNN on CIFAR-10 (PyTorch)

Experimenting with a baseline CNN architecture on the CIFAR-10 dataset as part of a comparative study of CNN architectures.

- **Framework:** PyTorch
- **Dataset:** CIFAR-10 (60,000 images, 10 classes)
- **Optimizer:** SGD | **Loss:** CrossEntropyLoss | **Epochs:** 30

## 1. Importing Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import time

## 2. Data Preprocessing
Normalizing pixel values to [-1, 1] range using mean and std of 0.5 per channel.

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5,0.5,0.5))
])

## 3. Loading CIFAR-10
Downloaded via torchvision. Batch size: 32, num_workers: 0.

In [3]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=0)

Files already downloaded and verified
Files already downloaded and verified


## 4. Data Exploration
Verifying image tensor shape and defining class labels.

In [4]:
image, label = train_data[0]

In [5]:
image.size()

torch.Size([3, 32, 32])

In [6]:
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

## 5. Model Architecture
2 Convolutional layers followed by 3 Fully Connected layers.

| Layer | Details |
|-------|---------|
| Conv1 | 3 → 12 filters, kernel 5x5 → MaxPool |
| Conv2 | 12 → 24 filters, kernel 5x5 → MaxPool |
| FC1 | 600 → 120 |
| FC2 | 120 → 84 |
| FC3 | 84 → 10 (output) |

- **Total Trainable Parameters:** 84,586
- **Activation:** ReLU

In [7]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3,12,5) # (12, 28, 28)
        self.pool = nn.MaxPool2d(2,2) # (12, 14, 14)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 10, 10) -> (24, 5, 5) -> Flatten(24*5*5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## 6. Loss Function and Optimizer
- **Loss:** CrossEntropyLoss
- **Optimizer:** SGD with lr=0.001 and momentum=0.9

In [8]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

## 7. Training
Training for 30 epochs. Loss decreases consistently across epochs indicating proper learning.

In [9]:
start_time = time.time()

for epoch in range(30):
    print(f'Training epoch {epoch}...')
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Loss: {running_loss / len(train_loader):.4f}")

end_time = time.time()
training_time = end_time - start_time
print(f"Training Time: {training_time:.2f} seconds")

Training epoch 0...
Loss: 2.2120
Training epoch 1...
Loss: 1.7754
Training epoch 2...
Loss: 1.5335
Training epoch 3...
Loss: 1.4115
Training epoch 4...
Loss: 1.3276
Training epoch 5...
Loss: 1.2475
Training epoch 6...
Loss: 1.1770
Training epoch 7...
Loss: 1.1151
Training epoch 8...
Loss: 1.0620
Training epoch 9...
Loss: 1.0126
Training epoch 10...
Loss: 0.9674
Training epoch 11...
Loss: 0.9298
Training epoch 12...
Loss: 0.8908
Training epoch 13...
Loss: 0.8542
Training epoch 14...
Loss: 0.8224
Training epoch 15...
Loss: 0.7922
Training epoch 16...
Loss: 0.7597
Training epoch 17...
Loss: 0.7352
Training epoch 18...
Loss: 0.7061
Training epoch 19...
Loss: 0.6794
Training epoch 20...
Loss: 0.6534
Training epoch 21...
Loss: 0.6294
Training epoch 22...
Loss: 0.6075
Training epoch 23...
Loss: 0.5814
Training epoch 24...
Loss: 0.5597
Training epoch 25...
Loss: 0.5390
Training epoch 26...
Loss: 0.5199
Training epoch 27...
Loss: 0.4981
Training epoch 28...
Loss: 0.4812
Training epoch 29...
Los

## 8. Saving the Model
Saving trained weights to disk for later evaluation.

In [10]:
torch.save(net.state_dict(), 'trained_net.pth' )

## 9. Loading the Model
Reloading saved weights for evaluation.

In [11]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth', weights_only=True))

<All keys matched successfully>

## 10. Evaluation

In [12]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item() 

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')

Accuracy: 68.26%


## 11. Results Summary

| Metric | Value |
|--------|-------|
| Test Accuracy | 68.26% |
| Trainable Parameters | 84,586 |
| Training Time | 646.52 seconds |

**Observation:** The baseline CNN achieves 68.26% accuracy on CIFAR-10. 
Performance is limited by the shallow architecture and absence of regularization techniques such as Batch Normalization or Dropout. 
Further experiments with deeper architectures like ResNet are expected to improve results significantly.